# Supplementary Figure S6 — species model errors

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
import numpy as np
from matplotlib.gridspec import GridSpec

cmp = "Greys"

MAX = 40
step = 10
cpauMax = 1000 + MAX
max_len = cpauMax

## Load the experimental dataset and fitted results

In [ ]:
import csv
archive = np.load(SAVED_RESULTS / 'fits_best_available.npz', allow_pickle=True)
cdc_x, cper_x = archive['CDC_X'], archive['CPER_X']
targets = archive['targets'].tolist()
fits = archive['fits']
fit_species = np.array([fit['species'] for fit in fits])
with (ROOT / 'data/species_names.csv').open(newline='') as source:
    aliases = {row['species_code']: row['figure_code'] for row in csv.DictReader(source)}
species_labels = np.array([aliases.get(code, code) for code in fit_species])

## Extract species curves and parameters

In [ ]:
cdc_Y = []
cper_Y = []
cdc_Ym = []
cper_Ym = []
cdc_Ys = []
cper_Ys = []
losses = []
inh_mults = []
ada_mults = []
for fit in fits:
    cdc_target, cper_target = targets[fit['species']]
    cdc_Y.append(cdc_target)
    cper_Y.append(cper_target)
    cdc_Ym.append(fit['cdc'])
    cper_Ym.append(fit['cper'])
    cdc_Ys.append(fit['seed_cdc'])
    cper_Ys.append(fit['seed_cper'])
    losses.append(fit['loss'])
    inh_mults.append(fit['inh_mult'])
    ada_mults.append(fit['ada_mult'])
cdc_Y = np.array(cdc_Y)
cper_Y = np.array(cper_Y)
cdc_Ys = np.array(cdc_Ys)
cper_Ys = np.array(cper_Ys)

## Cluster the endpoint duty-cycle responses

In [ ]:
num_clusters = 3
Z = linkage(pdist(cdc_Y[:,-2:], metric="euclidean"), method="average", optimal_ordering=True,)
labels = fcluster(Z, t=num_clusters, criterion="maxclust") - 1
cut = (Z[-num_clusters, 2] + Z[-num_clusters + 1, 2]) / 2

tree = dendrogram(Z, color_threshold=cut, no_plot=True, orientation='left', labels=species_labels)
order = tree['leaves']

## Calculate errors
Gray curves show the experimental targets; black curves show the fitted model. Blue bars are the AN1 divisive-normalization multiplier, and red bars are the AN1 inhibitory-filter multiplier. All rows share the same parameter scale.

In [ ]:
mses = []

for cnt, row in enumerate(order):
    mse = np.mean((cper_Y[row] - cper_Ym[row])**2) + np.mean((cdc_Y[row] - cdc_Ym[row])**2)
    mses.append(mse)

## Plot combined mean-squared errors

In [ ]:
species = species_labels

plt.figure(figsize=(4, 2))
plt.bar(range(len(mses)), mses[::-1], facecolor='lightgrey')
plt.ylabel('MSE')
plt.xticks(range(len(mses)), labels=species[order[::-1]], rotation=90)
plt.yticks([0, 0.1, 0.2, 0.3])
plt.xlim(-1, 14)
plt.ylim(0, max(0.32, 1.1 * max(mses)))

plt.savefig(f'{FIGURES}/fig_S6.pdf', bbox_inches='tight', facecolor=None)